In [1]:
!pip install -r requirements.txt


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import logging
import time
from datetime import datetime
from botocore.exceptions import ClientError

from strands import Agent, tool
from strands.hooks import (
    AgentInitializedEvent, 
    MessageAddedEvent, 
    AfterInvocationEvent,
    HookProvider, 
    HookRegistry
)
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

In [3]:
# Setup logging to see what the agent is doing
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("food-agent")

# Configuration
REGION = os.getenv('AWS_REGION', 'us-east-1')
USER_ID = "food-lover-001"  # User ID for short and long term memory using AWS Agent Core
SESSION_ID = f"food_chat_{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Region: {REGION}")
print(f"User ID: {USER_ID}")
print(f"Session ID: {SESSION_ID}")

Region: us-east-1
User ID: food-lover-001
Session ID: food_chat_20260306211247


In [4]:
# Initialize Memory Client
client = MemoryClient(region_name=REGION)
memory_name = "FoodAgentMemory"
memory_id = None

# Define memory strategy for food preferences
# Everything goes to short term memory first, then Agent Core moves preferences to long term memory (one time task)
strategies = [
    {
        StrategyType.USER_PREFERENCE.value: {
            "name": "FoodPreferences",
            "description": "Captures food preferences including cuisines, dietary restrictions, favorite dishes, and specific foods the user likes or dislikes",
            "namespaces": ["user/{actorId}/food_preferences"]
        }
    }
]

try:
    # Create memory resource
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=strategies,
        description="Memory for food recommendation agent - stores user food preferences",
        event_expiry_days=7,  # Keep preferences for a week
        max_wait=300,
        poll_interval=10
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # Memory already exists - retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"✅ Memory already exists. Using: {memory_id}")
    else:
        raise e

print(f"\n📝 Save this memory_id for future sessions: {memory_id}")

2026-03-06 21:12:50,322 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-06 21:12:50,437 - INFO - Initialized MemoryClient for control plane: us-east-1, data plane: us-east-1
2026-03-06 21:12:51,318 - ERROR - Failed to create memory: An error occurred (ValidationException) when calling the CreateMemory operation: Validation failed during CreateMemory: Memory with name FoodAgentMemory already exists
2026-03-06 21:12:51,688 - INFO - ✅ Memory already exists. Using: FoodAgentMemory-wZABcd31To



📝 Save this memory_id for future sessions: FoodAgentMemory-wZABcd31To


In [5]:
class FoodMemoryHookProvider(HookProvider):
    """Automatic memory management for food agent"""
    
    def __init__(self, memory_client: MemoryClient, memory_id: str):
        self.memory_client = memory_client
        self.memory_id = memory_id
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load food preferences when agent starts"""
        try:
            actor_id = event.agent.state.get("actor_id")
            if not actor_id:
                logger.warning("Missing actor_id in agent state")
                return
            
            namespace = f"user/{actor_id}/food_preferences"
            
            # Retrieve stored food preferences (querying food preferences genres directory)
            preferences = self.memory_client.retrieve_memories(
                memory_id=self.memory_id,
                namespace=namespace,
                query="food preferences cuisines dietary restrictions favorites",
                top_k=3
            )
            
            if preferences:
                # Format preferences for context
                pref_texts = []
                for pref in preferences:
                    if isinstance(pref, dict):
                        content = pref.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            if text:
                                pref_texts.append(f"- {text}")
                
                if pref_texts:
                    context = "\n".join(pref_texts)
                    event.agent.system_prompt += f"\n\n## User's Food Preferences (from previous conversations):\n{context}"
                    logger.info(f"✅ Loaded {len(pref_texts)} food preferences")
            else:
                logger.info("No previous food preferences found - starting fresh!")
                    
        except Exception as e:
            logger.error(f"Error loading preferences: {e}")
    
    def on_after_invocation(self, event: AfterInvocationEvent):
        """Save conversation after each interaction"""
        try:
            messages = event.agent.messages
            # Save to memory after two interactions
            if len(messages) < 2:
                return
                
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            
            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id")
                return
            
            # Get the last user message and assistant response
            user_msg = None
            assistant_msg = None
            
            for msg in reversed(messages):
                if msg["role"] == "assistant" and not assistant_msg:
                    content = msg.get("content", [])
                    if content and isinstance(content[0], dict) and "text" in content[0]:
                        assistant_msg = content[0]["text"]
                elif msg["role"] == "user" and not user_msg:
                    content = msg.get("content", [])
                    if content and isinstance(content[0], dict) and "text" in content[0]:
                        if "toolResult" not in content[0]:
                            user_msg = content[0]["text"]
                            break
            
            if user_msg and assistant_msg:
                # Save the conversation turn to short term memory
                self.memory_client.create_event(
                    memory_id=self.memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=[(user_msg, "USER"), (assistant_msg, "ASSISTANT")]
                )
                logger.info("💾 Saved conversation to memory")
                
        except Exception as e:
            logger.error(f"Error saving conversation: {e}")
    
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)
        logger.info("✅ Food memory hooks registered")

In [6]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException

@tool
def search_food(query: str, max_results: int = 5) -> str:
    """Search for food information, recipes, cuisines, or restaurant recommendations.
    
    Args:
        query: Search query about food (e.g., \"best authentic italian pasta recipes\")
        max_results: Maximum number of results to return
    
    Returns:
        Search results with food information
    """
    try:
        results = DDGS().text(f"{query} food recipe restaurant", region="us-en", max_results=max_results)
        if not results:
            return "No results found."
        
        formatted = []
        for i, r in enumerate(results, 1):
            formatted.append(f"{i}. {r.get('title', 'No title')}\n   {r.get('body', '')}")
        
        return "\n\n".join(formatted)
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("✅ Food search tool ready")

2026-03-06 21:13:02,389 - INFO - ✅ Food search tool ready


In [7]:
def create_food_agent(user_id: str, session_id: str):
    """Create a food recommendation agent with memory"""
    
    system_prompt = f"""You are a concise food assistant. Help users discover new foods & remember their preferences. Date: {datetime.today().strftime("%Y-%m-%d")}"""
    
    # Create memory hooks
    memory_hooks = FoodMemoryHookProvider(client, memory_id)
    
    # Create agent
    agent = Agent(
        name="FoodieBuddy",
        model="us.anthropic.claude-3-5-haiku-20241022-v1:0",
        system_prompt=system_prompt,
        hooks=[memory_hooks],
        tools=[search_food],
        state={"actor_id": user_id, "session_id": session_id}
    )
    
    return agent

# Create the agent
food_agent = create_food_agent(USER_ID, SESSION_ID)
logger.info("✅ Food agent created with memory!")

import time
import logging
logger = logging.getLogger("food-agent")

def safe_invoke(agent, user_input, max_retries=3):
    """Invokes the agent with rate limiting and fallback mechanisms."""
    fallbacks = [
        "us.anthropic.claude-3-5-haiku-20241022-v1:0",
        "us.anthropic.claude-3-haiku-20240307-v1:0",
        "us.anthropic.claude-3-5-sonnet-20241022-v2:0"
    ]
    
    time.sleep(1) # Client-side rate limit spacing
    
    for attempt in range(max_retries):
        try:
            return agent(user_input)
        except Exception as e:
            error_str = str(e).lower()
            if "throttl" in error_str or "429" in error_str or "rate" in error_str:
                logger.warning(f"⚠️ Throttled (Attempt {attempt+1}/{max_retries}). Error: {str(e)[:50]}")
                if attempt < len(fallbacks):
                    fallback_model = fallbacks[attempt]
                    logger.info(f"🔄 Switching to fallback model: {fallback_model}")
                    agent.model = fallback_model
                    time.sleep(2 ** attempt)
                else:
                    raise e
            else:
                raise e


2026-03-06 21:13:07,434 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-06 21:13:07,521 - INFO - ✅ Food memory hooks registered
2026-03-06 21:13:08,487 - INFO - Retrieved 3 memories from namespace: user/food-lover-001/food_preferences
2026-03-06 21:13:08,488 - INFO - ✅ Loaded 3 food preferences
2026-03-06 21:13:08,488 - INFO - ✅ Food agent created with memory!


In [8]:
# Tell the agent about your preferences
print("You: I really love spicy food, especially South Indian cuisines! But I have a Non-veg allergy.")
print("\nAgent: ", end="")
food_agent("I really love spicy food, especially South Indian cuisines! But I have a Non-veg allergy.")

2026-03-06 21:14:38,095 - INFO - Creating Strands MetricsClient


You: I really love spicy food, especially South Indian cuisines! But I have a Non-veg allergy.

Agent: I'll help you find some delicious spicy South Indian vegetarian dishes that will satisfy your love for spicy food while accommodating your dietary restrictions. Let me search for some great vegetarian South Indian recipes.
Tool #1: search_food


2026-03-06 21:14:42,025 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=Spicy%20vegetarian%20South%20Indian%20dishes%20and%20recipes%20food%20recipe%20restaurant 200
2026-03-06 21:14:42,178 - INFO - response: https://grokipedia.com/api/typeahead?query=Spicy+vegetarian+South+Indian+dishes+and+recipes+food+recipe+restaurant&limit=1 200
2026-03-06 21:14:43,745 - INFO - response: https://www.mojeek.com/search?q=Spicy+vegetarian+South+Indian+dishes+and+recipes+food+recipe+restaurant 200


Great! Based on the search results and your preferences, I can recommend some spicy South Indian vegetarian dishes that you might enjoy:

1. Spicy Vegetable Curries:
- Vegetable Korma
- Spicy Vegetable Masala
- Avial (mixed vegetable curry with a spicy kick)

2. Iconic Vegetarian Dishes:
- Masala Dosa (spicy potato-filled crispy crepe)
- Vegetable Biryani
- Sambar (spicy lentil and vegetable stew)

3. Spicy Side Dishes:
- Vegetable Poriyal (spicy stir-fried vegetables)
- Milagai Bajji (spicy chili fritters)
- Rasam (a spicy, tangy soup)

These dishes are completely vegetarian, so they'll work perfectly with your non-veg allergy. They're also known for their bold, spicy flavors that you love from South Indian cuisine. Would you like me to find a specific recipe for any of these dishes?

2026-03-06 21:14:50,621 - INFO - Created event: 0000001772811890145#5fcb9583
2026-03-06 21:14:50,622 - INFO - 💾 Saved conversation to memory


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Great! Based on the search results and your preferences, I can recommend some spicy South Indian vegetarian dishes that you might enjoy:\n\n1. Spicy Vegetable Curries:\n- Vegetable Korma\n- Spicy Vegetable Masala\n- Avial (mixed vegetable curry with a spicy kick)\n\n2. Iconic Vegetarian Dishes:\n- Masala Dosa (spicy potato-filled crispy crepe)\n- Vegetable Biryani\n- Sambar (spicy lentil and vegetable stew)\n\n3. Spicy Side Dishes:\n- Vegetable Poriyal (spicy stir-fried vegetables)\n- Milagai Bajji (spicy chili fritters)\n- Rasam (a spicy, tangy soup)\n\nThese dishes are completely vegetarian, so they'll work perfectly with your non-veg allergy. They're also known for their bold, spicy flavors that you love from South Indian cuisine. Would you like me to find a specific recipe for any of these dishes?"}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'search_food': ToolMetrics(tool={'toolUs

In [9]:
print("\nYou: Can you give me a recommendation for tonight?")
print("\nAgent: ", end="")
food_agent("Can you give me a recommendation for tonight?")


You: Can you give me a recommendation for tonight?

Agent: Let me search for a delicious and spicy South Indian vegetarian dish that would be perfect for tonight's dinner.
Tool #2: search_food


2026-03-06 21:15:30,913 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=Best%20easy%20spicy%20South%20Indian%20vegetarian%20dinner%20recipe%20Masala%20Dosa%20food%20recipe%20restaurant 200
2026-03-06 21:15:31,349 - INFO - response: https://grokipedia.com/api/typeahead?query=Best+easy+spicy+South+Indian+vegetarian+dinner+recipe+Masala+Dosa+food+recipe+restaurant&limit=1 200
2026-03-06 21:15:32,260 - INFO - response: https://search.brave.com/search?q=Best+easy+spicy+South+Indian+vegetarian+dinner+recipe+Masala+Dosa+food+recipe+restaurant&source=web 200


I recommend Vegetable Masala Dosa for tonight! Here's why it's perfect for you:
- Completely vegetarian (safe for your non-veg allergy)
- Spicy and flavorful (matches your love for spicy food)
- A classic South Indian dish
- Can be made relatively easily at home

Quick Recipe Highlights:
- Main components: Crispy dosa (rice and lentil crepe)
- Filling: Spicy potato and vegetable masala
- Spice level: Can be adjusted to your preference
- Typical cooking time: Around 45 minutes
- Served with spicy chutneys and sambar for extra heat

Would you like me to provide a detailed recipe for Vegetable Masala Dosa? It's a great option that ticks all your boxes - vegetarian, South Indian, and packed with spicy flavors!

2026-03-06 21:15:38,358 - INFO - Created event: 0000001772811937966#55d153cb
2026-03-06 21:15:38,361 - INFO - 💾 Saved conversation to memory


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I recommend Vegetable Masala Dosa for tonight! Here's why it's perfect for you:\n- Completely vegetarian (safe for your non-veg allergy)\n- Spicy and flavorful (matches your love for spicy food)\n- A classic South Indian dish\n- Can be made relatively easily at home\n\nQuick Recipe Highlights:\n- Main components: Crispy dosa (rice and lentil crepe)\n- Filling: Spicy potato and vegetable masala\n- Spice level: Can be adjusted to your preference\n- Typical cooking time: Around 45 minutes\n- Served with spicy chutneys and sambar for extra heat\n\nWould you like me to provide a detailed recipe for Vegetable Masala Dosa? It's a great option that ticks all your boxes - vegetarian, South Indian, and packed with spicy flavors!"}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'search_food': ToolMetrics(tool={'toolUseId': 'tooluse_VKEL42pPS36OpSwjEMoSmn', 'name': 'search_food', 'input': {'query': 'B

In [10]:
# Check SHORT-TERM MEMORY (Raw Conversations)
print("\nSHORT-TERM MEMORY (Raw Conversations)")
print("=" * 60)
events = client.list_events(memory_id=memory_id, actor_id=USER_ID, session_id=SESSION_ID)
if events:
    for i, event in enumerate(events, 1):
        print(f"\n--- Event {i} ---")
        print(event)
else:
    print("No events found. Memory ID:", memory_id)


SHORT-TERM MEMORY (Raw Conversations)


2026-03-06 21:15:58,428 - INFO - Retrieved total of 2 events



--- Event 1 ---
{'memoryId': 'FoodAgentMemory-wZABcd31To', 'actorId': 'food-lover-001', 'sessionId': 'food_chat_20260306211247', 'eventId': '0000001772811937966#55d153cb', 'eventTimestamp': datetime.datetime(2026, 3, 6, 21, 15, 37, 966000, tzinfo=tzlocal()), 'payload': [{'conversational': {'content': {'text': 'Can you give me a recommendation for tonight?'}, 'role': 'USER'}}, {'conversational': {'content': {'text': "I recommend Vegetable Masala Dosa for tonight! Here's why it's perfect for you:\n- Completely vegetarian (safe for your non-veg allergy)\n- Spicy and flavorful (matches your love for spicy food)\n- A classic South Indian dish\n- Can be made relatively easily at home\n\nQuick Recipe Highlights:\n- Main components: Crispy dosa (rice and lentil crepe)\n- Filling: Spicy potato and vegetable masala\n- Spice level: Can be adjusted to your preference\n- Typical cooking time: Around 45 minutes\n- Served with spicy chutneys and sambar for extra heat\n\nWould you like me to provide 

In [11]:
# Check LONG-TERM MEMORY (Background Processed Preferences)
print("\nLONG-TERM MEMORY PREFERENCES:")
print("=" * 50)
try:
    preferences = client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"user/{USER_ID}/food_preferences",
        query="food preferences cuisines dietary restrictions favorites",
        top_k=3
    )

    if preferences:
        for i, pref in enumerate(preferences, 1):
            if isinstance(pref, dict):
                content = pref.get('content', {})
                if isinstance(content, dict):
                    text = content.get('text', '')
                    if text:
                        print(f"{i}. {text}")
    else:
        print("No preferences extracted yet. It could take 30-60 seconds to process in the background.")
except Exception as e:
    print(f"Could not retrieve from long-term memory: {e}")


LONG-TERM MEMORY PREFERENCES:


2026-03-06 21:16:14,118 - INFO - Retrieved 3 memories from namespace: user/food-lover-001/food_preferences


1. {"context":"The user explicitly mentioned having a non-veg allergy.","preference":"Has a non-veg allergy (requires vegetarian options)","categories":["food","dietary restrictions","health"]}
2. {"context":"The user explicitly mentioned having a peanut allergy.","preference":"Has a peanut allergy","categories":["food","health","dietary restrictions","allergies"]}
3. {"context":"The user explicitly stated that they really love spicy food, especially Indian cuisines and Thai cuisines.","preference":"Loves spicy food, especially Indian and Thai cuisines","categories":["food","cuisine","spice level"]}


## Simulating a New Session
Let's simulate coming back later. We will create a new agent instance (like opening the app again) with the same `USER_ID` and memory, but a new `SESSION_ID`. We'll see if it remembers the user's preferences.

In [12]:
# Wait a few seconds to let background processes digest the memory
import time
print("Waiting for memory processing to finish...")
time.sleep(10)

# Simulate a new session
SESSION_ID_2 = f"food_chat_{datetime.now().strftime('%Y%m%d%H%M%S')}_2"
print(f"New Session ID: {SESSION_ID_2}")

# Create a new agent instance for the new session
food_agent_2 = create_food_agent(USER_ID, SESSION_ID_2)

Waiting for memory processing to finish...


2026-03-06 21:16:45,719 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-06 21:16:45,823 - INFO - ✅ Food memory hooks registered


New Session ID: food_chat_20260306211645_2


2026-03-06 21:16:46,274 - INFO - Retrieved 3 memories from namespace: user/food-lover-001/food_preferences
2026-03-06 21:16:46,275 - INFO - ✅ Loaded 3 food preferences


In [ ]:
# Test if the new agent remembers the preferences (peanut allergy, spicy food, Thai)
print("You: I'm hungry! What should I get for dinner tonight?")
print("\nAgent: ", end="")
food_agent_2("I'm hungry! What should I get for dinner tonight? Do you remember what I like and can't eat?")

You: I'm hungry! What should I get for dinner tonight?

Agent: I recall your specific dietary requirements and preferences! Based on our previous conversations, you:
1. Have a non-veg allergy (so we'll need vegetarian options)
2. Have a peanut allergy (which means we'll avoid peanuts)
3. Love spicy food, especially from Indian (particularly South Indian) and Thai cuisines

Let me help you find a delicious dinner that meets these requirements. I'll search for some spicy vegetarian recipes that should be perfect for you.
Tool #1: search_food


2026-03-06 21:17:11,866 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=spicy%20vegetarian%20South%20Indian%20dinner%20recipes%20food%20recipe%20restaurant 200
2026-03-06 21:17:11,883 - INFO - response: https://grokipedia.com/api/typeahead?query=spicy+vegetarian+South+Indian+dinner+recipes+food+recipe+restaurant&limit=1 200
2026-03-06 21:17:12,904 - INFO - response: https://yandex.com/search/site/?text=spicy+vegetarian+South+Indian+dinner+recipes+food+recipe+restaurant&web=1&searchid=7138332 200


Great! I see some promising options. How about a South Indian vegetarian thali (meal platter) that's sure to be spicy and exciting? This would include multiple vegetarian dishes like:
- Spicy vegetable sambar
- Vegetable curry
- Rasam (a tangy, spicy soup)
- Rice
- Various vegetable side dishes

Would you like me to find a specific recipe for this type of meal? I'll make sure to look for something that avoids peanuts and is 100% vegetarian. What do you think?

2026-03-06 21:17:17,376 - INFO - Created event: 0000001772812036924#f01cb8f9
2026-03-06 21:17:17,377 - INFO - 💾 Saved conversation to memory


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Great! I see some promising options. How about a South Indian vegetarian thali (meal platter) that's sure to be spicy and exciting? This would include multiple vegetarian dishes like:\n- Spicy vegetable sambar\n- Vegetable curry\n- Rasam (a tangy, spicy soup)\n- Rice\n- Various vegetable side dishes\n\nWould you like me to find a specific recipe for this type of meal? I'll make sure to look for something that avoids peanuts and is 100% vegetarian. What do you think?"}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'search_food': ToolMetrics(tool={'toolUseId': 'tooluse_O1ZA0mkeo4Gw8Tdh81f0Hs', 'name': 'search_food', 'input': {'query': 'spicy vegetarian South Indian dinner recipes', 'max_results': 3}}, call_count=1, success_count=1, error_count=0, total_time=1.5753390789031982)}, cycle_durations=[3.971510887145996], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_c